# AEGIS-Pipe — Cell 1: immutable configuration and raw-data audit

This cell performs only lightweight reads. It never loads a complete WFS file. The independent experimental unit is the **physical pipe** (B–E), so all channels from a held-out pipe must remain together in every later split.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import platform
import re
import shutil
import sys

import pandas as pd
from IPython.display import display

# -----------------------------------------------------------------------------
# Project paths — edit only RAW if the dataset is moved.
# -----------------------------------------------------------------------------
RAW = Path(r"D:\Pipeline RUL Data\data\raw")
PROJECT = RAW.parents[1]                 # D:\Pipeline RUL Data
INTERIM = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"
CACHE = PROJECT / "cache"
RUNS = PROJECT / "runs"
FIGURES = PROJECT / "figures_main"

for directory in (INTERIM, PROCESSED, CACHE, RUNS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class ExperimentConfig:
    pipe_ids: tuple[str, ...] = ("B", "C", "D", "E")
    active_ae_channels: tuple[int, ...] = (5, 6, 7, 8)
    nominal_sample_rate_hz: int = 1_000_000
    sample_dtype: str = "<i2"             # hypothesis; verify before decoding
    record_length_dtype: str = "<u2"      # verified from the supplied header
    stream_record_id: int = 0xAE
    header_probe_bytes: int = 4096
    split_group: str = "pipe_id"           # never split by channel/window
    random_seed: int = 20260907


CFG = ExperimentConfig()

if not RAW.is_dir():
    raise FileNotFoundError(f"Raw-data directory does not exist: {RAW}")

found = {path.stem.upper(): path for path in RAW.glob("*.wfs")}
missing = [pipe_id for pipe_id in CFG.pipe_ids if pipe_id not in found]
if missing:
    raise FileNotFoundError(f"Missing WFS files for pipes: {missing}")

files = [found[pipe_id] for pipe_id in CFG.pipe_ids]
extra = sorted(set(found).difference(CFG.pipe_ids))
if extra:
    print(f"Note: ignoring additional WFS stems: {extra}")

rows = []
for path in files:
    with path.open("rb") as stream:
        header = stream.read(CFG.header_probe_bytes)

    version_match = re.search(rb"Version\s+V([0-9.]+)", header)
    version = version_match.group(1).decode("ascii") if version_match else None
    rows.append(
        {
            "pipe_id": path.stem.upper(),
            "file": path.name,
            "size_bytes": path.stat().st_size,
            "size_GiB": path.stat().st_size / 2**30,
            "express8_signature": b"Express-8" in header,
            "software_version": version,
            "header_sha256_4KiB": hashlib.sha256(header).hexdigest(),
        }
    )

manifest = pd.DataFrame(rows).set_index("pipe_id")
manifest_path = RUNS / "raw_wfs_manifest.csv"
config_path = RUNS / "experiment_config.json"
manifest.to_csv(manifest_path)
config_path.write_text(
    json.dumps(
        {
            **asdict(CFG),
            "raw_directory": str(RAW),
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "python": sys.version.split()[0],
            "platform": platform.platform(),
        },
        indent=2,
    ),
    encoding="utf-8",
)

disk = shutil.disk_usage(PROJECT)
total_gib = manifest["size_GiB"].sum()
free_gib = disk.free / 2**30

display(manifest[["file", "size_GiB", "express8_signature", "software_version"]].round({"size_GiB": 2}))
print(f"\nRaw total: {total_gib:,.2f} GiB")
print(f"Free space on {PROJECT.drive or PROJECT.anchor}: {free_gib:,.2f} GiB")
print(f"Manifest: {manifest_path}")
print(f"Config:   {config_path}")
print("\nSplit invariant: pipe_id is the independent unit; channels 5–8 stay together.")

assert manifest["express8_signature"].all(), "At least one file lacks the Express-8 signature."
assert manifest["software_version"].notna().all(), "Could not recover the WFS software version."
assert manifest.index.is_unique, "Duplicate physical pipe IDs detected."


## Cell 2 — bounded WFS framing and waveform-layout probe

This cell reads three 8 MiB regions from **B.wfs** (start, middle, and end). It discovers record boundaries, checks the repeated `0xAE` framing, infers the likely waveform tail, proposes channel/timestamp fields, and writes only a small JSON report. It does not export raw samples.

In [ ]:
# Cell 2: bounded, read-only reverse-engineering probe for Express-8 WFS
from collections import Counter
from dataclasses import dataclass
import math
import warnings

import numpy as np

PROBE_FILE = files[0]                    # B.wfs only
REGION_BYTES = 8 * 2**20               # 8 MiB per region
ALIGN_SCAN_BYTES = 256 * 2**10         # alignment must be found here
MAX_RECORDS_PER_REGION = 256
MIN_AE_CHAIN = 12
MAX_PAYLOAD_BYTES = 65_535             # uint16 record-length ceiling
STREAM_ID = CFG.stream_record_id


@dataclass(frozen=True)
class WFSFrame:
    relative_offset: int
    payload_length: int
    record_id: int
    payload: bytes


def _payload_length(blob: bytes, offset: int) -> int:
    return int.from_bytes(blob[offset : offset + 2], byteorder="little", signed=False)


def _ae_chain_length(blob: bytes, start: int, limit: int = 64) -> int:
    """Count consecutive, correctly framed 0xAE records from start."""
    position = start
    count = 0
    while count < limit and position + 3 <= len(blob):
        length = _payload_length(blob, position)
        end = position + 2 + length
        if not (1 <= length <= MAX_PAYLOAD_BYTES) or end > len(blob):
            break
        if blob[position + 2] != STREAM_ID:
            break
        count += 1
        position = end
    return count


def find_ae_alignment(blob: bytes) -> tuple[int, int]:
    """Find a boundary supported by a long chain, not a coincidental 0xAE byte."""
    search_stop = min(len(blob) - 1, ALIGN_SCAN_BYTES)
    search_from = 2
    best = (-1, -1)  # (chain length, relative offset)

    while search_from < search_stop:
        id_position = blob.find(bytes((STREAM_ID,)), search_from, search_stop)
        if id_position < 0:
            break
        candidate = id_position - 2
        if candidate >= 0:
            length = _payload_length(blob, candidate)
            if 1 <= length <= MAX_PAYLOAD_BYTES and candidate + 2 + length <= len(blob):
                chain = _ae_chain_length(blob, candidate)
                if chain > best[0]:
                    best = (chain, candidate)
                if chain >= MIN_AE_CHAIN:
                    return candidate, chain
        search_from = id_position + 1

    raise RuntimeError(
        f"No trustworthy 0x{STREAM_ID:02X} record chain found; best chain={best[0]} "
        f"at relative offset={best[1]}. Do not increase the scan blindly."
    )


def parse_frames(blob: bytes, start: int, max_records: int) -> list[WFSFrame]:
    frames: list[WFSFrame] = []
    position = start
    while len(frames) < max_records and position + 3 <= len(blob):
        length = _payload_length(blob, position)
        end = position + 2 + length
        if not (1 <= length <= MAX_PAYLOAD_BYTES) or end > len(blob):
            break
        payload = bytes(blob[position + 2 : end])
        frames.append(WFSFrame(position, length, payload[0], payload))
        position = end
    return frames


def read_region(path: Path, start: int, size: int) -> bytes:
    with path.open("rb", buffering=0) as stream:
        stream.seek(start)
        return stream.read(size)


file_size = PROBE_FILE.stat().st_size
region_starts = {
    "start": 0,
    "middle": max(0, file_size // 2 - REGION_BYTES // 2),
    "end": max(0, file_size - REGION_BYTES),
}

all_frames: list[tuple[int, WFSFrame]] = []
region_reports = []
for region_name, region_start in region_starts.items():
    blob = read_region(PROBE_FILE, region_start, REGION_BYTES)
    alignment, chain = find_ae_alignment(blob)
    frames = parse_frames(blob, alignment, MAX_RECORDS_PER_REGION)
    all_frames.extend((region_start + frame.relative_offset, frame) for frame in frames)

    region_reports.append(
        {
            "region": region_name,
            "requested_offset": region_start,
            "aligned_offset": region_start + alignment,
            "alignment_skip_bytes": alignment,
            "verified_ae_chain": chain,
            "records_parsed": len(frames),
            "record_ids": {f"0x{k:02X}": int(v) for k, v in Counter(f.record_id for f in frames).items()},
            "payload_lengths": {str(k): int(v) for k, v in Counter(f.payload_length for f in frames).most_common(8)},
        }
    )

all_frames.sort(key=lambda item: item[0])
ae_frames = [frame for _, frame in all_frames if frame.record_id == STREAM_ID]
if not ae_frames:
    raise RuntimeError("The bounded probes contained no 0xAE records.")

length_counts = Counter(frame.payload_length for frame in ae_frames)
typical_payload_length, typical_count = length_counts.most_common(1)[0]
typical_frames = [frame for frame in ae_frames if frame.payload_length == typical_payload_length]

# For an int16 waveform, a power-of-two sample count should leave a short metadata prefix.
layout_candidates = []
for exponent in range(6, 16):
    sample_count = 2**exponent
    metadata_bytes = typical_payload_length - 2 * sample_count
    if 8 <= metadata_bytes <= 256:
        layout_candidates.append((sample_count, metadata_bytes))

if not layout_candidates:
    raise RuntimeError(
        f"Payload length {typical_payload_length} has no plausible int16/power-of-two layout."
    )

waveform_samples, metadata_bytes = min(layout_candidates, key=lambda item: item[1])
metadata = np.vstack(
    [np.frombuffer(frame.payload[:metadata_bytes], dtype=np.uint8) for frame in typical_frames]
)


def infer_channel_candidates(meta: np.ndarray) -> list[dict]:
    candidates = []
    n_rows, n_columns = meta.shape
    for width in (1, 2):
        for offset in range(0, n_columns - width + 1):
            if width == 1:
                raw_values = [int(value) for value in meta[:, offset]]
            else:
                raw_values = [
                    int.from_bytes(row[offset : offset + width].tobytes(), "little")
                    for row in meta
                ]

            for base in (0, 1):
                lower, upper = base, base + 7
                if not all(lower <= value <= upper for value in raw_values):
                    continue
                unique = sorted(set(raw_values))
                if len(unique) < 4:
                    continue
                normalized = [value + 1 if base == 0 else value for value in raw_values]
                transitions = [
                    ((right - left) % 8) == 1
                    for left, right in zip(normalized[:-1], normalized[1:])
                ]
                counts = Counter(normalized)
                balance = min(counts.values()) / max(counts.values())
                coverage = len(counts) / 8
                cycle_fraction = float(np.mean(transitions)) if transitions else 0.0
                endpoint_bonus = 0.05 * ((lower in unique) + (upper in unique))
                score = 0.50 * coverage + 0.30 * cycle_fraction + 0.20 * balance + endpoint_bonus
                score -= 0.02 * (width - 1)
                candidates.append(
                    {
                        "offset": offset,
                        "width_bytes": width,
                        "stored_base": base,
                        "unique_stored_values": unique,
                        "coverage": coverage,
                        "cycle_fraction": cycle_fraction,
                        "balance": balance,
                        "score": score,
                        "first_24_channels": normalized[:24],
                    }
                )
    return sorted(candidates, key=lambda item: item["score"], reverse=True)


channel_candidates = infer_channel_candidates(metadata)
if not channel_candidates:
    raise RuntimeError("No plausible 1–8 or 0–7 channel field was found in the metadata prefix.")
channel_field = channel_candidates[0]


def decode_channel(payload: bytes, field: dict) -> int:
    start = field["offset"]
    stop = start + field["width_bytes"]
    stored = int.from_bytes(payload[start:stop], "little")
    return stored + 1 if field["stored_base"] == 0 else stored


def infer_monotone_integer_fields(meta: np.ndarray, top_k: int = 10) -> list[dict]:
    """Rank possible clock/counter fields; final semantics remain unconfirmed."""
    candidates = []
    for width in (4, 8):
        for offset in range(0, meta.shape[1] - width + 1):
            values = [
                int.from_bytes(row[offset : offset + width].tobytes(), "little")
                for row in meta
            ]
            if len(set(values)) < 4:
                continue
            deltas = [right - left for left, right in zip(values[:-1], values[1:])]
            nondecreasing = sum(delta >= 0 for delta in deltas) / len(deltas)
            if nondecreasing < 0.98:
                continue
            positive = [delta for delta in deltas if delta > 0]
            repeat_fraction = sum(delta == 0 for delta in deltas) / len(deltas)
            span = max(values) - min(values)
            score = nondecreasing + 0.15 * repeat_fraction + 0.01 * math.log10(span + 1)
            candidates.append(
                {
                    "offset": offset,
                    "width_bytes": width,
                    "nondecreasing_fraction": nondecreasing,
                    "repeat_fraction": repeat_fraction,
                    "unique_values": len(set(values)),
                    "span": span,
                    "median_positive_delta": float(np.median(positive)) if positive else None,
                    "first_values": values[:4],
                    "last_values": values[-4:],
                    "score": score,
                }
            )
    return sorted(candidates, key=lambda item: item["score"], reverse=True)[:top_k]


clock_candidates = infer_monotone_integer_fields(metadata)

# Decode only the waveform tail of the bounded sample records.
block_stats: dict[int, list[dict]] = {channel: [] for channel in range(1, 9)}
for frame in typical_frames:
    channel = decode_channel(frame.payload, channel_field)
    if channel not in block_stats:
        continue
    samples_le = np.frombuffer(
        frame.payload, dtype="<i2", count=waveform_samples, offset=metadata_bytes
    ).astype(np.float64)
    samples_be = np.frombuffer(
        frame.payload, dtype=">i2", count=waveform_samples, offset=metadata_bytes
    ).astype(np.float64)
    block_stats[channel].append(
        {
            "mean": float(samples_le.mean()),
            "std": float(samples_le.std()),
            "ptp": float(samples_le.max() - samples_le.min()),
            "zero_fraction": float(np.mean(samples_le == 0)),
            "saturation_fraction": float(np.mean((samples_le == -32768) | (samples_le == 32767))),
            "big_to_little_std_ratio": float(samples_be.std() / max(samples_le.std(), 1e-12)),
        }
    )

channel_rows = []
for channel, records in block_stats.items():
    if not records:
        continue
    std_values = np.array([record["std"] for record in records])
    ptp_values = np.array([record["ptp"] for record in records])
    channel_rows.append(
        {
            "channel": channel,
            "blocks": len(records),
            "std_min": float(std_values.min()),
            "std_median": float(np.median(std_values)),
            "std_max": float(std_values.max()),
            "ptp_median": float(np.median(ptp_values)),
            "low_variance_fraction_std_lt_3": float(np.mean(std_values < 3.0)),
            "zero_fraction_mean": float(np.mean([record["zero_fraction"] for record in records])),
            "saturation_fraction_mean": float(np.mean([record["saturation_fraction"] for record in records])),
            "big_to_little_std_ratio_median": float(
                np.median([record["big_to_little_std_ratio"] for record in records])
            ),
        }
    )

channel_summary = pd.DataFrame(channel_rows).set_index("channel").sort_index()
region_summary = pd.DataFrame(region_reports).set_index("region")
channel_candidate_table = pd.DataFrame(channel_candidates[:8])
clock_candidate_table = pd.DataFrame(clock_candidates)
metadata_hex = [frame.payload[:metadata_bytes].hex(" ") for frame in typical_frames[:12]]

schema_report = {
    "probe_version": 1,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "file": str(PROBE_FILE),
    "file_size_bytes": file_size,
    "bytes_read_total": len(region_reports) * REGION_BYTES,
    "stream_record_id": f"0x{STREAM_ID:02X}",
    "regions": region_reports,
    "typical_payload_length": typical_payload_length,
    "typical_payload_observations": typical_count,
    "layout_candidates": [
        {"waveform_samples": samples, "metadata_bytes": prefix}
        for samples, prefix in layout_candidates
    ],
    "selected_layout_hypothesis": {
        "sample_dtype": "<i2",
        "waveform_samples": waveform_samples,
        "metadata_bytes": metadata_bytes,
    },
    "channel_field_candidates": channel_candidates[:8],
    "selected_channel_field_hypothesis": channel_field,
    "clock_field_candidates": clock_candidates,
    "channel_summary": channel_summary.reset_index().to_dict(orient="records"),
    "metadata_hex_first_12": metadata_hex,
}
schema_report_path = RUNS / "wfs_schema_probe.json"
schema_report_path.write_text(json.dumps(schema_report, indent=2), encoding="utf-8")

if typical_payload_length != 8220:
    warnings.warn(f"Expected the reported 8220-byte payload, observed {typical_payload_length}.")

print(f"Probe file: {PROBE_FILE.name}")
print(f"Read only {schema_report['bytes_read_total'] / 2**20:.1f} MiB from {file_size / 2**30:.2f} GiB.")
print(f"Dominant 0xAE payload: {typical_payload_length} bytes ({typical_count} bounded observations)")
print(f"Layout hypothesis: {metadata_bytes}-byte metadata + {waveform_samples} int16 samples")
print(
    "Channel-field hypothesis: "
    f"offset={channel_field['offset']}, width={channel_field['width_bytes']}, "
    f"stored_base={channel_field['stored_base']}"
)
print(f"Schema report: {schema_report_path}\n")

display(region_summary[["aligned_offset", "alignment_skip_bytes", "verified_ae_chain", "records_parsed", "record_ids", "payload_lengths"]])
display(channel_candidate_table)
display(channel_summary.round(4))
display(clock_candidate_table)

print("First 12 metadata prefixes:")
for index, value in enumerate(metadata_hex, start=1):
    print(f"{index:02d}: {value}")


### Stop-and-check gate

Before building the full streaming decoder, verify that the dominant payload is 8220 bytes, the proposed layout is 28 metadata bytes plus 4096 signed 16-bit samples, the inferred channel sequence is credible, and channels 5–8 have the expected activity. Paste the Cell 2 output for confirmation.